# GDV – World Cup 2022 Goal Build-up Analysis

## Research question
Wie entstehen Tore bei der FIFA WM 2022? Entstehen Tore eher durch schnelle Angriffe mit wenigen Pässen oder durch längere Passsequenzen?

## Use case
Ein Trainer, Analyst oder Fussballfan möchte verstehen, wie Tore entstehen: Wie viele Pässe braucht ein Team bis zum Tor? Wo beginnt der Angriff? Über welche Zonen läuft der Ball?

Dieses Notebook baut die Datenbasis und erzeugt die wichtigsten GDV-Figuren.

In [ ]:
from pathlib import Path
import ast
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsbombpy import sb

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

GOAL_BUILDUPS_FILE = PROCESSED_DIR / 'goal_buildups.csv'
GOAL_PASSES_FILE = PROCESSED_DIR / 'goal_buildup_passes.csv'

COMPETITION_ID = 43
SEASON_ID = 106

print(PROJECT_ROOT)

## 1. Helper functions
StatsBomb speichert Positionen als Listen im Format `[x, y]`. Wir extrahieren daraus Start- und Endpunkte von Pässen.

In [ ]:
def parse_location(value):
    if isinstance(value, list) and len(value) >= 2:
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list) and len(parsed) >= 2:
                return parsed
        except Exception:
            return None
    return None

def get_x(value):
    loc = parse_location(value)
    return loc[0] if isinstance(loc, list) else np.nan

def get_y(value):
    loc = parse_location(value)
    return loc[1] if isinstance(loc, list) else np.nan

def seconds_total(row):
    return int(row['minute']) * 60 + int(row.get('second', 0))

## 2. Build goal build-up dataset
Logik: Für jedes Tor suchen wir die gleiche Possession des gleichen Teams. Dann zählen wir alle erfolgreichen Pässe vor dem Torschuss. Wenn ein Unterbruch passiert, beginnt StatsBomb normalerweise eine neue Possession. Genau deshalb ist `possession` für diese Fragestellung passend.

In [ ]:
def build_goal_sequences(force_rebuild=False):
    if GOAL_BUILDUPS_FILE.exists() and GOAL_PASSES_FILE.exists() and not force_rebuild:
        print('Existing processed files found. Loading CSV files...')
        return pd.read_csv(GOAL_BUILDUPS_FILE), pd.read_csv(GOAL_PASSES_FILE)

    matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
    all_buildups = []
    all_passes = []

    for match_idx, match in matches.reset_index(drop=True).iterrows():
        match_id = int(match['match_id'])
        home_team = match['home_team']
        away_team = match['away_team']
        match_label = f'{home_team} vs {away_team}'
        print(f'[{match_idx + 1}/{len(matches)}] {match_label}')

        try:
            events = sb.events(match_id=match_id)
        except Exception as exc:
            print(f'Could not load match {match_id}: {exc}')
            continue

        required_cols = ['id','index','period','timestamp','minute','second','team','player','type','possession','location','pass_end_location','pass_outcome','shot_outcome','play_pattern']
        for col in required_cols:
            if col not in events.columns:
                events[col] = np.nan

        events = events.sort_values(['period', 'index']).copy()
        events['event_seconds'] = events.apply(seconds_total, axis=1)
        events['x'] = events['location'].apply(get_x)
        events['y'] = events['location'].apply(get_y)
        events['end_x'] = events['pass_end_location'].apply(get_x)
        events['end_y'] = events['pass_end_location'].apply(get_y)

        goals = events[(events['type'] == 'Shot') & (events['shot_outcome'].astype(str).str.lower() == 'goal')].copy()

        for goal_number, goal in goals.reset_index(drop=True).iterrows():
            team = goal['team']
            possession = goal['possession']
            goal_index = goal['index']
            goal_seconds = goal['event_seconds']
            build_up_id = f'{match_id}_{goal_number + 1}'

            same_possession = events[(events['possession'] == possession) & (events['team'] == team) & (events['index'] <= goal_index)].copy()
            if same_possession.empty:
                continue

            successful_passes = same_possession[(same_possession['type'] == 'Pass') & (same_possession['pass_outcome'].isna() | (same_possession['pass_outcome'].astype(str).str.lower() == 'nan'))].copy()

            if successful_passes.empty:
                first_event = same_possession.iloc[0]
                start_x = first_event['x']
                start_y = first_event['y']
                start_seconds = first_event['event_seconds']
            else:
                first_pass = successful_passes.iloc[0]
                start_x = first_pass['x']
                start_y = first_pass['y']
                start_seconds = first_pass['event_seconds']

            all_buildups.append({
                'build_up_id': build_up_id,
                'match_id': match_id,
                'match_date': match.get('match_date'),
                'match_label': match_label,
                'team': team,
                'goal_player': goal.get('player'),
                'goal_minute': goal.get('minute'),
                'goal_second': goal.get('second'),
                'period': goal.get('period'),
                'possession': possession,
                'play_pattern': goal.get('play_pattern'),
                'passes_before_goal': len(successful_passes),
                'duration_seconds': max(0, goal_seconds - start_seconds),
                'start_x': start_x,
                'start_y': start_y,
                'goal_x': goal.get('x'),
                'goal_y': goal.get('y'),
            })

            for pass_order, p in successful_passes.reset_index(drop=True).iterrows():
                all_passes.append({
                    'build_up_id': build_up_id,
                    'match_id': match_id,
                    'match_label': match_label,
                    'team': team,
                    'goal_player': goal.get('player'),
                    'goal_minute': goal.get('minute'),
                    'pass_order': pass_order + 1,
                    'player': p.get('player'),
                    'minute': p.get('minute'),
                    'second': p.get('second'),
                    'x': p.get('x'),
                    'y': p.get('y'),
                    'end_x': p.get('end_x'),
                    'end_y': p.get('end_y'),
                })

        time.sleep(0.1)

    buildups = pd.DataFrame(all_buildups)
    passes = pd.DataFrame(all_passes)
    buildups.to_csv(GOAL_BUILDUPS_FILE, index=False)
    passes.to_csv(GOAL_PASSES_FILE, index=False)
    return buildups, passes

buildups, passes = build_goal_sequences(force_rebuild=False)
print(buildups.shape, passes.shape)

## 3. Quick data check
Wir prüfen, ob die Daten sinnvoll aussehen. Wichtig sind: Anzahl Tore, Anzahl Pässe vor Toren, Angriffsdauer und Teams.

In [ ]:
display(buildups.head())
print('Goals analysed:', len(buildups))
print('Teams:', buildups['team'].nunique())
print('Passes in goal buildups:', len(passes))
display(buildups[['passes_before_goal', 'duration_seconds']].describe())

## 4. Pitch helper
Diese Funktion zeichnet ein einfaches StatsBomb-Spielfeld mit 120x80 Koordinaten.

In [ ]:
def draw_pitch(ax):
    ax.plot([0, 120], [0, 0], color='black', linewidth=1)
    ax.plot([0, 120], [80, 80], color='black', linewidth=1)
    ax.plot([0, 0], [0, 80], color='black', linewidth=1)
    ax.plot([120, 120], [0, 80], color='black', linewidth=1)
    ax.plot([60, 60], [0, 80], color='black', linewidth=1)
    ax.add_patch(plt.Circle((60, 40), 10, fill=False, color='black', linewidth=1))
    ax.plot([0, 18], [18, 18], color='black', linewidth=1)
    ax.plot([18, 18], [18, 62], color='black', linewidth=1)
    ax.plot([18, 0], [62, 62], color='black', linewidth=1)
    ax.plot([120, 102], [18, 18], color='black', linewidth=1)
    ax.plot([102, 102], [18, 62], color='black', linewidth=1)
    ax.plot([102, 120], [62, 62], color='black', linewidth=1)
    ax.set_xlim(0, 120)
    ax.set_ylim(80, 0)
    ax.set_aspect('equal')
    ax.axis('off')

## Figure 1 – Wie viele Pässe gehen Toren voraus?
Diese Grafik beantwortet die Kernfrage direkt. Links sieht man schnelle Angriffe mit wenigen Pässen, rechts längere Passsequenzen.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(buildups['passes_before_goal'], bins=range(0, int(buildups['passes_before_goal'].max()) + 2), edgecolor='black')
ax.set_title('Figure 1: Passes before goals')
ax.set_xlabel('Successful passes before goal')
ax.set_ylabel('Number of goals')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'goal_fig01_passes_before_goals.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 2 – Welche Teams brauchen im Schnitt wie viele Pässe bis zum Tor?
Diese Grafik zeigt Team-Unterschiede. Teams mit wenigen Pässen wirken direkter, Teams mit vielen Pässen eher geduldiger im Aufbau.

In [ ]:
team_summary = (
    buildups.groupby('team')
    .agg(goals=('build_up_id', 'count'), avg_passes=('passes_before_goal', 'mean'), median_passes=('passes_before_goal', 'median'), avg_duration=('duration_seconds', 'mean'))
    .reset_index()
)
team_summary = team_summary[team_summary['goals'] >= 2].sort_values('avg_passes', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(team_summary['team'], team_summary['avg_passes'])
ax.set_title('Figure 2: Average passes before goal by team')
ax.set_xlabel('Average successful passes before goal')
ax.set_ylabel('Team')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'goal_fig02_avg_passes_by_team.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 3 – Wo beginnen Torangriffe?
Die Heatmap zeigt die Startorte der Torangriffe. So erkennt man, ob erfolgreiche Angriffe oft tief, zentral oder hoch auf dem Feld beginnen.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
draw_pitch(ax)
heat = ax.hist2d(buildups['start_x'], buildups['start_y'], bins=[12, 8], range=[[0, 120], [0, 80]], cmap='Reds', alpha=0.75)
ax.set_title('Figure 3: Start locations of goal build-ups')
fig.colorbar(heat[3], ax=ax, fraction=0.025, pad=0.02, label='Number of goal build-ups')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'goal_fig03_start_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 4 – Passverkehr vor Toren
Diese Pass Map zeigt die Pässe, die direkt in Tor-Possessions vorkamen. Sie ist eine kompakte Übersicht über typische Ballwege vor Toren.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
draw_pitch(ax)
sample_passes = passes.dropna(subset=['x', 'y', 'end_x', 'end_y']).copy()
for _, p in sample_passes.iterrows():
    ax.arrow(p['x'], p['y'], p['end_x'] - p['x'], p['end_y'] - p['y'], length_includes_head=True, head_width=1.2, alpha=0.18, color='blue')
ax.set_title('Figure 4: Passes in goal build-ups')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'goal_fig04_pass_flow_map.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 5 – Case Study: Eine Torsequenz Schritt für Schritt
Diese Figur zeigt eine einzelne Tor-Possession. Die Zahlen zeigen die Reihenfolge der Pässe. Das ist die statische Grundlage für die spätere IVI-Animation.

In [ ]:
case = buildups.sort_values('passes_before_goal', ascending=False).iloc[0]
case_id = case['build_up_id']
case_passes = passes[passes['build_up_id'] == case_id].sort_values('pass_order')

fig, ax = plt.subplots(figsize=(10, 6))
draw_pitch(ax)
for _, p in case_passes.iterrows():
    ax.arrow(p['x'], p['y'], p['end_x'] - p['x'], p['end_y'] - p['y'], length_includes_head=True, head_width=1.5, color='blue', alpha=0.75)
    ax.text(p['x'], p['y'], str(int(p['pass_order'])), fontsize=9, weight='bold')
ax.scatter([case['goal_x']], [case['goal_y']], marker='*', s=350, color='gold', edgecolor='black', label='Goal')
ax.set_title(f"Figure 5: Case study goal build-up – {case['team']} / {case['goal_player']} / {case['passes_before_goal']} passes")
ax.legend(loc='lower center')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'goal_fig05_case_study_sequence.png', dpi=300, bbox_inches='tight')
plt.show()

display(case_passes[['pass_order', 'player', 'minute', 'second', 'x', 'y', 'end_x', 'end_y']])

## Short GDV interpretation
Diese Analyse beantwortet die Frage, ob Tore eher durch schnelle oder längere Angriffe entstehen. Die wichtigsten Variablen sind `passes_before_goal`, `duration_seconds`, `start_x/start_y` und die Passkoordinaten.

Für IVI ist besonders die Case-Study wichtig: Dort kann später ein Step-Slider gebaut werden, der Pass 1, Pass 2, Pass 3 bis zum Tor zeigt.